installing

In [1]:
!pip -q install chromadb sentence-transformers \
  pandas numpy requests scikit-learn \
  shap pypdf openai matplotlib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requi

importing libraries

In [2]:
import os
import json
import requests
import pandas as pd
import numpy as np

from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

NASA dataset collector

In [3]:
DATA_DIR = Path("/content/ASTRABIO/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

ACCESSIONS = [569, 570, 571, 574, 575]

def get_nasa_metadata(accession):

    url = (
        "https://osdr.nasa.gov/osdr/data/"
        f"osd/meta/{accession}"
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    data = response.json()

    output = DATA_DIR / f"OSD_{accession}.json"

    output.write_text(
        json.dumps(data, indent=2)
    )

    return data

metadata_collection = {}

for accession in ACCESSIONS:
    try:
        metadata_collection[accession] = (
            get_nasa_metadata(accession)
        )
        print(f"Successfully retrieved OSD-{accession}")
    except Exception as e:
        print(f"Could not retrieve OSD-{accession}: {e}")

Successfully retrieved OSD-569
Successfully retrieved OSD-570
Successfully retrieved OSD-571
Successfully retrieved OSD-574
Successfully retrieved OSD-575


RAG initailor

In [4]:
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

chroma_client = chromadb.PersistentClient(
    path="/content/ASTRABIO/vector_db"
)

collection = chroma_client.get_or_create_collection(
    name="astrabio_nasa_research"
)

def add_document(text, accession, category):

    embedding = embedding_model.encode(
        text
    ).tolist()

    doc_id = f"OSD-{accession}-{collection.count()}"

    collection.add(
        ids=[doc_id],
        documents=[text],
        embeddings=[embedding],
        metadatas=[{
            "accession": str(accession),
            "category": category,
            "organism_verified": "false"
        }]
    )

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Research metadology

In [5]:
for accession, data in metadata_collection.items():

    text = json.dumps(data, ensure_ascii=False)

    add_document(
        text=text[:15000],
        accession=accession,
        category="NASA study metadata"
    )

print("NASA metadata indexed:", collection.count())

NASA metadata indexed: 5


Scientific data retrivel

In [6]:
def search_astrabio(question, top_k=5):

    embedding = embedding_model.encode(
        question
    ).tolist()

    results = collection.query(
        query_embeddings=[embedding],
        n_results=min(top_k, collection.count())
    )

    return results

question = (
    "What does NASA research investigate "
    "about human immune changes during spaceflight?"
)

results = search_astrabio(question)

for document in results["documents"][0]:
    print(document[:1500])
    print("-" * 60)

{"hits": 1, "input": "569", "study": {"OSD-569": {"comments": [], "ontologySourceReferences": [{"comments": [], "description": "Experimental Factor Ontology", "file": "http://data.bioontology.org/ontologies/EFO", "name": "EFO", "version": "132"}, {"comments": [], "description": "", "file": "", "name": "NCIT", "version": ""}, {"comments": [], "description": "Ontology for Biomedical Investigations", "file": "http://data.bioontology.org/ontologies/OBI", "name": "OBI", "version": "29"}, {"comments": [], "description": "", "file": "", "name": "CRO", "version": ""}, {"comments": [], "description": "", "file": "", "name": "MESH", "version": ""}, {"comments": [], "description": "", "file": "", "name": "RBO", "version": ""}], "people": [], "publications": [], "studies": [{"assays": [{"characteristicCategories": [], "comments": [], "dataFiles": [{"@id": "#data/rawdatafile-140116241711056", "comments": [], "name": "Not Available", "type": "Raw Data File"}], "filename": "a_OSD-569_complete-blood-c

LLM connector

In [7]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

def astrabio_companion(question):

    results = search_astrabio(question)

    evidence = "\n\n".join(
        results["documents"][0]
    )

    response = client.responses.create(
        model="YOUR_AVAILABLE_MODEL",
        instructions="""
        You are the ASTRABIO scientific assistant.

        Answer using the supplied NASA research context.

        Distinguish human and animal research.
        Explain uncertainty.
        Do not diagnose medical conditions.
        Do not prescribe medication.
        Never invent scientific evidence.
        Mention study accession numbers where available.
        """,
        input=f"""
        Question:
        {question}

        Retrieved scientific evidence:
        {evidence}
        """
    )

    return response.output_text

print(
    astrabio_companion(
        "Explain the relevance of immune changes "
        "during long-duration spaceflight."
    )
)

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

ML temp

In [ ]:
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

df = pd.read_csv(filename)

print(df.head())
print(df.columns.tolist())

inspection

In [ ]:
# CHANGE THESE TO MATCH YOUR DATASET

FEATURES = ["gene_1", "gene_2", "gene_3"]
TARGET = "mission_phase_label"
GROUP = "subject_id"

X = df[FEATURES]
y = df[TARGET]
groups = df[GROUP]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
)

model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

predictions = model.predict(X.iloc[test_idx])

print(
    classification_report(
        y.iloc[test_idx],
        predictions
    )
)

SHAP explainablity

In [ ]:
import shap

rf = model.named_steps["randomforestclassifier"]

X_test = X.iloc[test_idx].fillna(
    X.iloc[train_idx].median()
)

explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(X_test)

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURES
)